# AI_Finance_Sec — 멀티턴 챗봇 오케스트레이션

이 노트북 하나에 P0/P1 데모용 챗봇 처리 전 과정을 구현합니다.

- 모델 선택: Mock / Ollama Local / OpenAI / Claude / Gemini / DeepSeek / Qwen Cloud
- 상태 정의: 대화, 위험 분석, RAG 문서, 대응 정책, 실행 추적
- 병렬 Node: 위험 분석 / 지식 검색 / 대응 정책
- Edge: 병렬 fan-out·fan-in, Tool 호출 조건 분기, 종료
- Tool: 보이스피싱 위험 분석, 금융 안전 가이드 검색, 대응 조치 추천
- Compile: LangGraph 실행 객체 생성
- Memory·Config: `thread_id`별 멀티턴 기억과 세션 격리

> 기본값은 API 키가 필요 없는 `mock`입니다. `ollama`도 localhost에서 API 키 없이 실행할 수 있습니다. 실제 클라우드 모델을 사용할 때만 Provider와 API 키를 선택합니다. ChatGPT Pro 구독과 OpenAI API 사용량은 별도이므로 실제 OpenAI 호출에는 API 키가 필요합니다.

## 1. 의존성 설치

최초 한 번 실행합니다. 설치 후 커널 재시작 안내가 나오면 재시작하고 다음 셀부터 실행합니다.

In [ ]:
%pip install -qU "langgraph>=1.0,<2" "langchain-core>=1.0,<2" "langchain-openai>=1.0,<2" "langchain-anthropic>=1.0,<2" "langchain-google-genai>=2.0,<5" "openai>=2.0,<3"

## 2. 공통 모듈과 실행 설정

`PROVIDER`만 바꾸면 실행 모델을 전환할 수 있습니다. 실제 서비스에서는 모델 ID를 환경변수로 관리하는 편이 안전합니다.

In [ ]:
from __future__ import annotations

import json
import operator
import os
import re
from getpass import getpass
from typing import Annotated, Any, Literal, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

Provider = Literal["mock", "ollama", "openai", "anthropic", "gemini", "deepseek", "qwen"]

# 데모 기본값: API 키 없이 전체 그래프 실행
PROVIDER: Provider = "mock"

MODEL_BY_PROVIDER = {
    "ollama": os.getenv("OLLAMA_MODEL", "qwen2.5:7b"),
    "openai": os.getenv("OPENAI_MODEL", "gpt-5.6-terra"),
    "anthropic": os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
    "deepseek": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
    "qwen": os.getenv("QWEN_MODEL", "qwen-plus"),
}
KEY_ENV_BY_PROVIDER = {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}
DEFAULT_THREAD_ID = "demo-user-001"

def prepare_provider(provider: Provider) -> None:
    if provider in {"mock", "ollama"}:
        detail = "결정론적 응답" if provider == "mock" else f"{os.getenv('OLLAMA_BASE_URL', 'http://127.0.0.1:11434/v1')} / {MODEL_BY_PROVIDER[provider]}"
        print(f"{provider} 모드: API 키 없이 실행합니다. ({detail})")
        return
    env_name = KEY_ENV_BY_PROVIDER[provider]
    if not os.getenv(env_name):
        os.environ[env_name] = getpass(f"{env_name} 입력(화면에 표시되지 않음): ")
    if not os.getenv(env_name):
        raise ValueError(f"{env_name}가 필요합니다.")
    print(f"{provider} / {MODEL_BY_PROVIDER[provider]} 모드가 준비됐습니다.")

prepare_provider(PROVIDER)

## 3. State 정의

`messages`는 `add_messages` reducer로 누적됩니다. `trace`도 병렬 Node 결과가 충돌하지 않도록 리스트 결합 reducer를 사용합니다.

In [ ]:
class ChatState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    user_input: str
    risk_analysis: dict[str, Any]
    rag_context: list[dict[str, str]]
    policy_result: dict[str, Any]
    final_answer: str
    provider: str
    turn_count: int
    trace: Annotated[list[str], operator.add]

print(ChatState.__annotations__)

## 4. Tool 정의

Tool은 병렬 분석 Node가 직접 사용하고, 실제 LLM에도 바인딩됩니다. 실제 금융기관 신고·지급정지 기능은 연결하지 않고 데모 권고만 반환합니다.

In [ ]:
RISK_RULES = {
    "기관사칭": {"keywords": ["검찰", "금감원", "수사관", "안전계좌"], "weight": 22},
    "긴급성": {"keywords": ["지금 즉시", "오늘 안에", "전화 끊지", "비밀"], "weight": 17},
    "금전요구": {"keywords": ["이체", "송금", "수수료", "현금", "입금"], "weight": 38},
    "앱설치": {"keywords": ["앱 설치", "원격제어", "apk", "링크 클릭"], "weight": 28},
    "가족빙자": {"keywords": ["자녀", "아들", "딸", "사고", "납치"], "weight": 19},
    "피해발생": {"keywords": ["이미 송금", "이미 이체", "송금했", "이체했", "보냈", "입금했"], "weight": 55},
}

GUIDANCE_DB = {
    "기관사칭": "수사기관과 금융기관은 전화로 안전계좌 이체를 요구하지 않습니다. 통화를 끊고 공식 대표번호로 직접 확인하세요.",
    "금전요구": "송금을 중단하고 계좌번호·통화기록·문자 등 증거를 보관하세요. 이미 송금했다면 금융회사와 112에 즉시 지급정지를 요청하세요.",
    "앱설치": "상대가 보낸 링크를 열거나 앱을 설치하지 마세요. 이미 설치했다면 네트워크를 차단하고 다른 안전한 기기로 금융회사에 연락하세요.",
    "가족빙자": "통화를 종료한 뒤 가족 본인과 다른 가족에게 별도로 연락해 안전을 확인하세요. 상대가 지정한 사람에게 돈을 전달하지 마세요.",
    "피해발생": "이미 송금했다면 추가 송금을 중단하고 금융회사와 112에 즉시 지급정지를 요청하세요. 계좌번호·이체내역·통화기록·문자를 보관하세요.",
    "일반": "의심스러운 요구에는 즉시 응하지 말고 공식 앱·대표번호를 통해 사실을 다시 확인하세요.",
}

@tool
def detect_phishing_risk(text: str) -> dict[str, Any]:
    """사용자 문장에서 보이스피싱 위험 신호를 탐지한다. 점수(0~100), 단계, 유형, 근거를 반환한다. 오류 시 error 필드를 반환한다."""
    normalized = re.sub(r"\s+", " ", text.lower()).strip()
    hits: list[dict[str, Any]] = []
    score = 0
    for category, rule in RISK_RULES.items():
        matched = [keyword for keyword in rule["keywords"] if keyword.lower() in normalized]
        if matched:
            added = min(rule["weight"] + 5 * (len(matched) - 1), 60)
            score += added
            hits.append({"category": category, "keywords": matched, "score_added": added})
    score = min(score, 100)
    level = "위험" if score >= 70 else "주의" if score >= 40 else "안전"
    primary_type = max(hits, key=lambda item: item["score_added"])["category"] if hits else "일반"
    return {"score": score, "level": level, "risk_type": primary_type, "evidence": hits}

@tool
def retrieve_safety_guidance(risk_type: str) -> list[dict[str, str]]:
    """위험 유형에 맞는 검증용 데모 금융 안전 가이드를 검색한다. title과 content 목록을 반환한다."""
    key = risk_type if risk_type in GUIDANCE_DB else "일반"
    return [{"title": f"{key} 대응 가이드", "content": GUIDANCE_DB[key], "source": "P0/P1 데모 지식베이스"}]

@tool
def recommend_response_actions(risk_score: int, already_transferred: bool = False) -> dict[str, Any]:
    """위험 점수와 송금 여부를 바탕으로 사용자에게 보여줄 비파괴적 대응 조치를 반환한다. 외부 신고를 실제 실행하지 않는다."""
    actions = ["공식 대표번호로 사실 확인"]
    if risk_score >= 40:
        actions = ["통화 즉시 종료", "추가 송금·앱 설치 중단", "통화·문자·계좌 증거 보관"] + actions
    if risk_score >= 70 or already_transferred:
        actions += ["112 상담·신고", "금융회사 콜센터에 지급정지 요청"]
    return {"actions": actions, "external_action_executed": False}

TOOLS = [detect_phishing_risk, retrieve_safety_guidance, recommend_response_actions]
TOOL_NODE = ToolNode(TOOLS)
[tool.name for tool in TOOLS]

## 5. 모델 생성 및 Tool 바인딩

OpenAI는 Responses API 사용을 명시합니다. Ollama는 localhost의 OpenAI-compatible `/v1` API를 API 키 없이 사용합니다. Qwen Cloud는 DashScope API 키가 필요한 별도 Provider입니다. 다른 Provider도 동일한 Tool 스키마를 바인딩합니다. Provider별 모델 ID는 환경변수로 덮어쓸 수 있습니다.

In [ ]:
def create_tool_bound_model(provider: Provider):
    if provider == "mock":
        return None

    model_id = MODEL_BY_PROVIDER[provider]
    if provider == "ollama":
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(
            model=model_id,
            api_key="ollama-local-no-key",
            base_url=os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434/v1"),
            temperature=0.1,
            timeout=float(os.getenv("OLLAMA_TIMEOUT_SECONDS", "45")),
            max_retries=1,
        )
    elif provider == "openai":
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(
            model=model_id,
            reasoning_effort="low",
            use_responses_api=True,
            timeout=60,
            max_retries=2,
        )
    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        model = ChatAnthropic(model=model_id, temperature=0, timeout=60, max_retries=2)
    elif provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        model = ChatGoogleGenerativeAI(model=model_id, temperature=0, timeout=60, max_retries=2)
    elif provider == "deepseek":
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(
            model=model_id,
            api_key=os.environ["DEEPSEEK_API_KEY"],
            base_url="https://api.deepseek.com",
            temperature=0,
            timeout=60,
            max_retries=2,
        )
    elif provider == "qwen":
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(
            model=model_id,
            api_key=os.environ["DASHSCOPE_API_KEY"],
            base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
            temperature=0.1,
        )
    else:
        raise ValueError(f"지원하지 않는 provider: {provider}")

    return model.bind_tools(TOOLS)

TOOL_BOUND_MODEL = create_tool_bound_model(PROVIDER)
print("Tool-bound model:", PROVIDER if TOOL_BOUND_MODEL else "mock deterministic responder")

## 6. Node 정의

`risk_agent`, `knowledge_agent`, `policy_agent`는 서로 독립적이므로 병렬 실행합니다. `assistant_orchestrator`가 결과를 통합하고, 실제 모델이 추가 Tool을 요청하면 Tool Node를 거쳐 다시 모델로 돌아옵니다.

In [ ]:
SYSTEM_PROMPT = """
당신은 AI_Finance_Sec 금융 보안 비서다.
- 사용자를 불안하게 만들지 말고 짧고 구체적인 한국어로 답한다.
- 탐지 근거와 즉시 할 일을 구분한다.
- 실제 신고·지급정지를 실행했다고 주장하지 않는다.
- 정보가 부족하면 필요한 사실을 한 번에 하나씩 확인한다.
- 제공된 위험 분석과 가이드가 부족할 때만 바인딩된 Tool을 사용한다.
""".strip()

def prepare_input(state: ChatState) -> dict[str, Any]:
    human_messages = [m for m in state.get("messages", []) if isinstance(m, HumanMessage)]
    user_input = state.get("user_input") or (str(human_messages[-1].content) if human_messages else "")
    if not user_input.strip():
        raise ValueError("사용자 입력이 비어 있습니다.")
    return {"user_input": user_input.strip(), "provider": PROVIDER, "trace": ["prepare_input"]}

def risk_agent(state: ChatState) -> dict[str, Any]:
    result = detect_phishing_risk.invoke({"text": state["user_input"]})
    return {"risk_analysis": result, "trace": ["risk_agent"]}

def _infer_risk_type(text: str) -> str:
    detected = detect_phishing_risk.invoke({"text": text})
    return str(detected["risk_type"])

def knowledge_agent(state: ChatState) -> dict[str, Any]:
    risk_type = _infer_risk_type(state["user_input"])
    docs = retrieve_safety_guidance.invoke({"risk_type": risk_type})
    return {"rag_context": docs, "trace": ["knowledge_agent"]}

def policy_agent(state: ChatState) -> dict[str, Any]:
    detected = detect_phishing_risk.invoke({"text": state["user_input"]})
    transferred = bool(re.search(r"이미.*(송금|이체)|보냈|입금했", state["user_input"]))
    policy = recommend_response_actions.invoke({"risk_score": detected["score"], "already_transferred": transferred})
    return {"policy_result": policy, "trace": ["policy_agent"]}

def _mock_answer(state: ChatState) -> str:
    risk = state["risk_analysis"]
    docs = state["rag_context"]
    actions = state["policy_result"]["actions"]
    evidence = [kw for item in risk["evidence"] for kw in item["keywords"]]
    evidence_text = " · ".join(evidence) if evidence else "명확한 고위험 키워드 없음"
    action_text = " → ".join(actions[:4])
    return (
        f"현재 위험도는 {risk['level']}({risk['score']}/100)로 판단됩니다. "
        f"탐지 근거: {evidence_text}.\n\n"
        f"지금 할 일: {action_text}.\n"
        f"안내: {docs[0]['content']}"
    )

def assistant_orchestrator(state: ChatState) -> dict[str, Any]:
    if PROVIDER == "mock":
        return {"messages": [AIMessage(content=_mock_answer(state))], "trace": ["assistant_orchestrator(mock)"]}

    context = {
        "risk_analysis": state.get("risk_analysis"),
        "rag_context": state.get("rag_context"),
        "policy_result": state.get("policy_result"),
    }
    prompt = SystemMessage(content=SYSTEM_PROMPT + "\n\n분석 컨텍스트:\n" + json.dumps(context, ensure_ascii=False))
    response = TOOL_BOUND_MODEL.invoke([prompt, *state.get("messages", [])])
    return {"messages": [response], "trace": [f"assistant_orchestrator({PROVIDER})"]}

def route_after_assistant(state: ChatState) -> Literal["tools", "finalize"]:
    last_message = state["messages"][-1]
    return "tools" if isinstance(last_message, AIMessage) and last_message.tool_calls else "finalize"

def finalize_response(state: ChatState) -> dict[str, Any]:
    last_message = state["messages"][-1]
    content = last_message.content
    if isinstance(content, list):
        content = "\n".join(str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in content)
    return {
        "final_answer": str(content),
        "turn_count": int(state.get("turn_count", 0)) + 1,
        "trace": ["finalize_response"],
    }

## 7. Edge 연결, Compile, Memory 정의

세 분석 Node를 병렬로 실행한 뒤 모두 완료되어야 오케스트레이터가 시작됩니다. `InMemorySaver`는 데모용이며 프로세스가 종료되면 초기화됩니다.

In [ ]:
workflow = StateGraph(ChatState)

workflow.add_node("prepare_input", prepare_input)
workflow.add_node("risk_agent", risk_agent)
workflow.add_node("knowledge_agent", knowledge_agent)
workflow.add_node("policy_agent", policy_agent)
workflow.add_node("assistant_orchestrator", assistant_orchestrator)
workflow.add_node("tools", TOOL_NODE)
workflow.add_node("finalize", finalize_response)

workflow.add_edge(START, "prepare_input")
workflow.add_edge("prepare_input", "risk_agent")
workflow.add_edge("prepare_input", "knowledge_agent")
workflow.add_edge("prepare_input", "policy_agent")
workflow.add_edge(["risk_agent", "knowledge_agent", "policy_agent"], "assistant_orchestrator")
workflow.add_conditional_edges(
    "assistant_orchestrator",
    route_after_assistant,
    {"tools": "tools", "finalize": "finalize"},
)
workflow.add_edge("tools", "assistant_orchestrator")
workflow.add_edge("finalize", END)

memory = InMemorySaver()
chat_graph = workflow.compile(checkpointer=memory)
print("Graph compiled with InMemorySaver")

## 8. 전체 Graph 확인

Mermaid 원문은 별도 렌더러 없이도 실행됩니다.

In [ ]:
print(chat_graph.get_graph().draw_mermaid())

## 9. Config 바인딩 및 멀티턴 호출 함수

동일한 `thread_id`를 사용하면 이전 대화가 이어지고, 다른 `thread_id`를 사용하면 별도 세션이 됩니다.

In [ ]:
def thread_config(thread_id: str = DEFAULT_THREAD_ID) -> dict[str, Any]:
    return {
        "configurable": {"thread_id": thread_id},
        "recursion_limit": 12,
        "tags": ["AI_Finance_Sec", PROVIDER],
        "metadata": {"provider": PROVIDER},
    }

def chat(message: str, thread_id: str = DEFAULT_THREAD_ID, show_trace: bool = True) -> str:
    config = thread_config(thread_id)
    result = chat_graph.invoke(
        {"messages": [HumanMessage(content=message)], "user_input": message},
        config=config,
    )
    if show_trace:
        print(f"thread={thread_id} / turn={result['turn_count']} / provider={result['provider']}")
        print("trace:", " → ".join(result.get("trace", [])))
    print("assistant:", result["final_answer"])
    return result["final_answer"]

## 10. 멀티턴 데모

아래 두 호출은 같은 `thread_id`를 사용하므로 두 번째 호출 시 첫 번째 대화가 Memory에 포함됩니다.

In [ ]:
chat("검찰 수사관이라며 지금 즉시 안전계좌로 이체하라고 합니다.")
print("\n" + "-" * 80 + "\n")
chat("이미 50만 원을 송금했어요. 지금 무엇부터 해야 하나요?")

## 11. Memory와 세션 격리 확인

In [ ]:
main_snapshot = chat_graph.get_state(thread_config(DEFAULT_THREAD_ID))
main_messages = main_snapshot.values.get("messages", [])
print("기본 세션 메시지 수:", len(main_messages))
print("기본 세션 turn_count:", main_snapshot.values.get("turn_count"))

chat("대출 수수료를 먼저 입금하라고 합니다.", thread_id="demo-user-002", show_trace=False)
other_snapshot = chat_graph.get_state(thread_config("demo-user-002"))
print("다른 세션 turn_count:", other_snapshot.values.get("turn_count"))

assert main_snapshot.values.get("turn_count") == 2
assert other_snapshot.values.get("turn_count") == 1
assert len(main_messages) > len(other_snapshot.values.get("messages", []))
print("멀티턴 기억 및 thread_id 세션 격리 검증 완료")

## 12. 프론트엔드·FastAPI 연결 시 사용 지점

실제 앱에서는 FastAPI의 `POST /api/chat`가 `message`와 `session_id`를 받은 뒤 아래처럼 호출하면 됩니다.

```python
result = await chat_graph.ainvoke(
    {"messages": [HumanMessage(content=request.message)], "user_input": request.message},
    config={"configurable": {"thread_id": request.session_id}},
)
return {"answer": result["final_answer"], "risk": result["risk_analysis"]}
```

P0/P1에서는 이 노트북의 `InMemorySaver`와 샘플 지식베이스를 사용합니다. P2에서는 영속 Checkpointer, 승인된 RAG 저장소, 금융사 레거시 API, 실시간 이벤트 입력으로 교체합니다.

## 13. 오프라인 RAGAS 평가(선택)

RAGAS는 실시간 판정기가 아니라 배포 전 회귀평가에만 사용합니다. `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`을 버전별로 비교하고, 위험 분류는 별도의 Macro F1·Recall·FPR로 평가합니다. 아래 데이터셋은 API 키가 준비된 평가 환경에서 `ragas.evaluate`에 입력할 수 있는 최소 형태입니다.

In [ ]:
RAGAS_EVALUATION_DATASET = [
    {
        "user_input": "검찰이 안전계좌로 송금하라고 합니다.",
        "retrieved_contexts": [GUIDANCE_DB["기관사칭"]],
        "response": "통화를 종료하고 공식 대표번호와 112로 확인하세요. 안전계좌로 송금하지 마세요.",
        "reference": "수사기관은 안전계좌 이체를 요구하지 않으므로 통화와 송금을 중단하고 공식 채널로 확인한다.",
    },
]
print("RAGAS 평가 레코드 수:", len(RAGAS_EVALUATION_DATASET))
print("실행은 별도 평가 환경에서 수행하며, 데모 런타임에는 포함하지 않습니다.")